# PWV02 : Seasonal Variation — Publication-quality figures

**Goal :** Demonstrate annual PWV variations at Cerro Pachon with
- Sinusoidal fit (period = 1 year) on MJD axis
- Year-folded (phase) plot Jan→Dec with one colour per year
- Error bars on PWV, markers/colours by filter
- Compact 2-panel layout with GridSpec (full timeline + seasonal phase)
- Lomb–Scargle periodogram as independent statistical evidence
- Seasonal statistics table (Summer/Autumn/Winter/Spring) + LaTeX export

**Author :** Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS  
**Created :** 2026-03-19  
**Kernel :** conda_py313

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.resetwarnings()
warnings.simplefilter('ignore')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from scipy.optimize import curve_fit
from astropy.time import Time

%matplotlib inline

# ── publication-style rc params ───────────────────────────────────────────────
mpl.rcParams.update({
    'figure.dpi'     : 150,
    'font.family'    : 'serif',
    'font.size'      : 11,
    'axes.labelsize' : 13,
    'axes.titlesize' : 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'axes.grid'      : True,
    'grid.alpha'     : 0.35,
    'axes.linewidth' : 1.1,
})

In [ ]:
from PWV00_parameters import (
    version_run, tag, extractedfilesdict, legendtag,
    PWV_FILTER_LIST, FLAG_PWVFILTERS,
    DumpConfig
)
from mysitcom.auxtel.qualitycuts import ParameterCutSelection, ParameterCutTools
from mysitcom.auxtel.pwv import normalize_column_data_bytarget_byfilter

DumpConfig()

In [ ]:
# ── output directories ────────────────────────────────────────────────────────
pathfigs = "figs_PWV02seas"
prefix   = "pwv02seas"
os.makedirs(pathfigs, exist_ok=True)
figtype  = ".pdf"   # vector for journal submission; change to .png for quick preview

In [ ]:
#Cut bad PWV from Merra2
PWVMIN = 0.
PWVMAX = 16.

---
## 1. Load data

In [ ]:
atmfilename   = extractedfilesdict[version_run]
inputfilename = atmfilename.split("/")[-1]
print(f"Loading: {atmfilename}")

if "parquet" in inputfilename:
    df_spec = pd.read_parquet(atmfilename)
else:
    specdata = np.load(atmfilename, allow_pickle=True)
    df_spec  = pd.DataFrame(specdata)

# ── Build Time column from DATE-OBS, exactly as in PWV01 ─────────────────────
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"], utc=True)
df_spec = df_spec.sort_values(by="Time", ascending=True).reset_index(drop=True)

print(f"Shape: {df_spec.shape}")
print(f"Time range: {df_spec['Time'].min()}  →  {df_spec['Time'].max()}")
print("Columns:", ", ".join(df_spec.columns))

In [ ]:
# ── rename Corentin run columns to standardised names ─────────────────────────
if "run2026_v02" in version_run:
    df_spec["chi2_ram"] = df_spec["CHI2_FIT"]
    df_spec["chi2_rum"] = df_spec["CHI2_FIT"]

# ── keep only PWV-sensitive filters ───────────────────────────────────────────
if FLAG_PWVFILTERS:
    df_spec = df_spec[df_spec["FILTER"].isin(PWV_FILTER_LIST)].copy()

# ── drop rows with missing PWV ────────────────────────────────────────────────
pwv_cols = ["PWV [mm]_ram", "PWV [mm]_rum", "PWV [mm]_err_ram", "PWV [mm]_err_rum"]
df_spec.dropna(subset=pwv_cols, inplace=True)

print(f"After filter selection: {len(df_spec)} rows")
print("Filters present:", df_spec["FILTER"].unique())

In [ ]:
# ── normalise chi2 per target per filter (needed for quality cuts) ─────────────
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="CHI2_FIT", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_ram", ext="norm")
df_spec, _ = normalize_column_data_bytarget_byfilter(
    df_spec, target_col="TARGET", filter_col="FILTER",
    feature_col="chi2_rum", ext="norm")

---
## 2. Apply quality cuts (tight)

In [ ]:
FLAG_LOOSE_CUTS = True
FLAG_TIGHT_CUTS = False

In [ ]:
pathdata      = "data_PWV01seas"   # reuse cuts from PWV01 notebook
if FLAG_LOOSE_CUTS:
    filename_cuts = f"{pathdata}/cuts_loose_finaldecision.json" 
    cutstype_name = "loose-cuts"
elif FLAG_TIGHT_CUTS: 
    filename_cuts = f"{pathdata}/cuts_tight_finaldecision.json" 
    cutstype_name = "tight-cuts"
else:
    filename_cuts = f"{pathdata}/cuts_finaldecision.json" 
    cutstype_name = "standard-cuts"

cuts           = ParameterCutTools.load_cuts_json(filename_cuts)
list_of_params = list(cuts.keys())

selector    = ParameterCutSelection(df_spec, params=list_of_params, id_col="id")
flags       = selector.apply_cuts(cuts)
df_selected = df_spec.merge(flags, on="id")
df_keep     = df_selected[df_selected["pass_all_cuts"]].copy()

print(f"Tight cuts: {len(df_keep)} / {len(df_spec)} observations kept")

---
## 3. Derived time columns

`Time` (UTC-aware datetime) was built from `DATE-OBS` in §1, exactly
as in PWV01.  Here we derive calendar fields and the MJD fractional-year
column used for the sinusoidal fit.

In [ ]:
# ── constants ─────────────────────────────────────────────────────────────────
MJD_J2000 = 51544.5   # 2000-01-01T12:00 UTC
YEAR_DAYS  = 365.25

# ── calendar fields from the existing Time column ─────────────────────────────
df_keep["year"]     = df_keep["Time"].dt.year
df_keep["month"]    = df_keep["Time"].dt.month
df_keep["doy"]      = df_keep["Time"].dt.day_of_year   # 1-366
df_keep["doy_norm"] = (df_keep["doy"] - 1) / 365.25    # 0→1  Jan→Dec

# ── fractional year since J2000 (sine-fit x-axis) ────────────────────────────
df_keep["mjd_year"] = (df_keep["MJD"] - MJD_J2000) / YEAR_DAYS

mjd_min = df_keep["MJD"].min()
mjd_max = df_keep["MJD"].max()

# ── best-estimate PWV and combined uncertainty ────────────────────────────────
df_keep["PWV"]     = 0.5 * (df_keep["PWV [mm]_ram"] + df_keep["PWV [mm]_rum"])
df_keep["PWV_err"] = np.sqrt(
    df_keep["PWV [mm]_err_ram"]**2 + df_keep["PWV [mm]_err_rum"]**2
)

years_present = sorted(df_keep["year"].dropna().astype(int).unique())
print("Years in dataset :", years_present)
print(f"MJD range        : {mjd_min:.1f} — {mjd_max:.1f}")
print(f"Date range       : {df_keep['Time'].min().date()}  →  {df_keep['Time'].max().date()}")

---
## 4. Sinusoidal fit  (period fixed = 1 yr)

In [ ]:
def pwv_annual_sine(t, A, B, phi):
    """Annual sinusoidal model.  PWV(t) = A + B*sin(2pi t + phi),  t in fractional years."""
    return A + B * np.sin(2.0 * np.pi * t + phi)


x_fit = df_keep["mjd_year"].values
y_fit = df_keep["PWV"].values
e_fit = df_keep["PWV_err"].values

mask = np.isfinite(x_fit) & np.isfinite(y_fit) & np.isfinite(e_fit) & (e_fit > 0)
x_fit, y_fit, e_fit = x_fit[mask], y_fit[mask], e_fit[mask]

p0 = [np.median(y_fit), 2.0, 0.0]
popt, pcov = curve_fit(
    pwv_annual_sine, x_fit, y_fit,
    p0=p0, sigma=e_fit, absolute_sigma=True,
    maxfev=10_000
)
perr = np.sqrt(np.diag(pcov))

A_fit, B_fit, phi_fit = popt
A_err, B_err, phi_err = perr

# month of PWV minimum  (when 2pi t + phi = -pi/2)
t_min_mod = ((-np.pi/2 - phi_fit) / (2*np.pi)) % 1.0
month_min  = 1 + t_min_mod * 12

MONTH_ABBR = ["Jan","Feb","Mar","Apr","May","Jun",
              "Jul","Aug","Sep","Oct","Nov","Dec"]

print("=== Annual sinusoidal fit ===")
print(f"  Offset    A = {A_fit:.3f} +/- {A_err:.3f} mm")
print(f"  Amplitude B = {B_fit:.3f} +/- {B_err:.3f} mm")
print(f"  Phase     phi = {phi_fit:.3f} +/- {phi_err:.3f} rad")
print(f"  PWV minimum ~ month {month_min:.1f}  ({MONTH_ABBR[int(month_min)-1]})")
print(f"  Peak-to-peak = {2*abs(B_fit):.2f} mm  ({2*abs(B_fit)/A_fit*100:.0f}% of mean)")

---
## 5. Monthly weighted statistics

In [ ]:
monthly = []
for m in range(1, 13):
    sub = df_keep[df_keep["month"] == m]
    if len(sub) == 0:
        monthly.append({"month": m, "pwv_mean": np.nan, "pwv_sem": np.nan, "n": 0})
        continue
    w   = 1.0 / (sub["PWV_err"].values**2 + 1e-12)
    mu  = np.average(sub["PWV"].values, weights=w)
    sem = 1.0 / np.sqrt(w.sum())
    monthly.append({"month": m, "pwv_mean": mu, "pwv_sem": sem, "n": len(sub)})

df_monthly = pd.DataFrame(monthly)
display(df_monthly)

---
## 6. Helpers: date axis & style maps

In [ ]:
def make_date_axis(ax_mjd, mjd_min, mjd_max, n_ticks=8):
    """Add a secondary top x-axis showing calendar dates (YYYY-MM)."""
    ax_date = ax_mjd.twiny()
    ax_date.set_xlim(ax_mjd.get_xlim())
    tick_mjds   = np.linspace(mjd_min, mjd_max, n_ticks)
    tick_labels = [
        Time(m, format='mjd', scale='utc').strftime('%Y-%m')
        for m in tick_mjds
    ]
    ax_date.set_xticks(tick_mjds)
    ax_date.set_xticklabels(tick_labels, rotation=30, ha='left', fontsize=9)
    ax_date.set_xlabel('Date (UTC)', fontsize=10, labelpad=6)
    return ax_date


# ── Filter styles ─────────────────────────────────────────────────────────────
FILTERS_OF_INTEREST = ["empty", "OG550_65mm_1", "FELH0600"]

filter_color  = {
    "empty"        : "#1f77b4",
    "OG550_65mm_1" : "#d62728",
    "FELH0600"     : "#2ca02c",
}
filter_marker = {
    "empty"        : "o",
    "OG550_65mm_1" : "s",
    "FELH0600"     : "^",
}
filter_label  = {
    "empty"        : "empty",
    "OG550_65mm_1" : "OG550",
    "FELH0600"     : "FELH0600",
}

# ── Year colours ──────────────────────────────────────────────────────────────
year_cmap  = mpl.colormaps.get_cmap("tab10").resampled(len(years_present))
year_color = {yr: year_cmap(i) for i, yr in enumerate(years_present)}

# ── Month tick positions ──────────────────────────────────────────────────────
month_ticks = [(m - 0.5) / 12.0 for m in range(1, 13)]
month_x     = (df_monthly["month"].values - 0.5) / 12.0

print("Filter colours:", filter_color)
print("Year colours:  ", {yr: mpl.colors.to_hex(c) for yr, c in year_color.items()})

---
## 7. Build fit curves (timeline + phase)

In [ ]:
# ── Full-timeline curve ───────────────────────────────────────────────────────
mjd_year_min = (mjd_min - MJD_J2000) / YEAR_DAYS
mjd_year_max = (mjd_max - MJD_J2000) / YEAR_DAYS
t_curve      = np.linspace(mjd_year_min, mjd_year_max, 2000)
mjd_curve    = t_curve * YEAR_DAYS + MJD_J2000
pwv_curve    = pwv_annual_sine(t_curve, *popt)

# 1-sigma propagated uncertainty band
dydA_c   = np.ones_like(t_curve)
dydB_c   = np.sin(2*np.pi*t_curve + phi_fit)
dydphi_c = B_fit * np.cos(2*np.pi*t_curve + phi_fit)
pwv_band = np.sqrt(
    (dydA_c * A_err)**2 + (dydB_c * B_err)**2 + (dydphi_c * phi_err)**2
)

# ── Phase (seasonal) curve: fraction-of-year 0 -> 1 ─────────────────────────
t_phase_norm = np.linspace(0, 1, 500)
t_phase_ref  = (Time("2023-01-01").mjd - MJD_J2000) / YEAR_DAYS + t_phase_norm
pwv_phase    = pwv_annual_sine(t_phase_ref, *popt)
dydA_p   = np.ones_like(t_phase_ref)
dydB_p   = np.sin(2*np.pi*t_phase_ref + phi_fit)
dydphi_p = B_fit * np.cos(2*np.pi*t_phase_ref + phi_fit)
pwv_phase_band = np.sqrt(
    (dydA_p * A_err)**2 + (dydB_p * B_err)**2 + (dydphi_p * phi_err)**2
)

---
## 8. Main figure — 2-panel GridSpec

- **Top** : PWV vs MJD — full timeline, sinusoidal fit, date axis
- **Bottom** : year-folded Jan -> Dec — all years superimposed, colour = year, marker = filter

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs  = GridSpec(2, 1, figure=fig, height_ratios=[1.4, 1.0], hspace=0.48)
ax_top = fig.add_subplot(gs[0])
ax_bot = fig.add_subplot(gs[1])

# == TOP PANEL — full timeline ==
for filt in FILTERS_OF_INTEREST:
    sub = df_keep[df_keep["FILTER"] == filt]
    if len(sub) == 0:
        continue
    ax_top.errorbar(
        sub["MJD"], sub["PWV"], yerr=sub["PWV_err"],
        fmt=filter_marker[filt], color=filter_color[filt],
        markersize=3, linewidth=0, elinewidth=0.6, capsize=1.5,
        alpha=0.55, label=filter_label[filt], zorder=3
    )

ax_top.plot(mjd_curve, pwv_curve, color="black", lw=2.0, zorder=5,
            label=(f"Annual sine fit\n"
                   f"$A={A_fit:.2f}\\pm{A_err:.2f}$ mm, "
                   f"$|B|={abs(B_fit):.2f}\\pm{B_err:.2f}$ mm"))
ax_top.fill_between(mjd_curve, pwv_curve - pwv_band, pwv_curve + pwv_band,
                    alpha=0.18, color="black", label=r"$\pm1\sigma$ fit")

ax_top.set_xlabel("MJD", fontsize=12)
ax_top.set_ylabel("PWV [mm]", fontsize=12)
ax_top.set_title(
    "Precipitable Water Vapour at Cerro Pachon — AuxTel/Spectractor",
    fontsize=12, pad=28)
ax_top.set_xlim(mjd_min - 30, mjd_max + 30)
ax_top.set_ylim(PWVMIN, PWVMAX)
ax_top.legend(loc="upper right", fontsize=9, framealpha=0.8)
make_date_axis(ax_top, mjd_min, mjd_max, n_ticks=10)

# == BOTTOM PANEL — year-folded ==
for yr in years_present:
    sub_yr = df_keep[df_keep["year"] == yr]
    for filt in FILTERS_OF_INTEREST:
        sub = sub_yr[sub_yr["FILTER"] == filt]
        if len(sub) == 0:
            continue
        ax_bot.errorbar(
            sub["doy_norm"], sub["PWV"], yerr=sub["PWV_err"],
            fmt=filter_marker[filt], color=year_color[yr],
            markersize=3, linewidth=0, elinewidth=0.5, capsize=1.2,
            alpha=0.55, zorder=3
        )

ax_bot.plot(t_phase_norm, pwv_phase, color="black", lw=2.0, zorder=5,
            label="Annual sine (mean year)")
ax_bot.fill_between(t_phase_norm, pwv_phase - pwv_phase_band,
                    pwv_phase + pwv_phase_band, alpha=0.15, color="black")
ax_bot.errorbar(
    month_x, df_monthly["pwv_mean"], yerr=df_monthly["pwv_sem"],
    fmt="D", color="darkorange", markersize=7,
    elinewidth=1.5, capsize=4, zorder=6, label="Monthly mean $\\pm$ s.e.m."
)

ax_bot.set_xticks(month_ticks)
ax_bot.set_xticklabels(MONTH_ABBR, fontsize=10)
ax_bot.set_xlim(0, 1)
ax_bot.set_ylim(PWVMIN, PWVMAX)
ax_bot.set_xlabel("Month of year", fontsize=12)
ax_bot.set_ylabel("PWV [mm]", fontsize=12)
ax_bot.set_title("Seasonal phase — all years superimposed", fontsize=12)

year_hdl = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=year_color[yr],
           markersize=8, label=str(yr))
    for yr in years_present
]
filt_hdl = [
    Line2D([0],[0], marker=filter_marker[f], color='gray',
           markerfacecolor='gray', markersize=7, linestyle='', label=filter_label[f])
    for f in FILTERS_OF_INTEREST if f in df_keep["FILTER"].values
]
extra_hdl = [
    Line2D([0],[0], color='black', lw=2, label='Annual sine fit'),
    Line2D([0],[0], marker='D', color='darkorange', markersize=7,
           linestyle='', label='Monthly mean'),
]
ax_bot.legend(
    handles=year_hdl + filt_hdl + extra_hdl,
    loc="upper right", fontsize=8.5,
    ncol=max(1, (len(years_present) + 4) // 4),
    framealpha=0.85
)

fit_text = (
    f"Sine fit:  $A = {A_fit:.2f}\\pm{A_err:.2f}$ mm,  "
    f"$|B| = {abs(B_fit):.2f}\\pm{B_err:.2f}$ mm,  "
    f"min $\\approx$ {MONTH_ABBR[int(month_min)-1]}"
)
fig.text(0.5, 0.005, fit_text, ha="center", fontsize=10,
         bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", ec="gray", lw=0.8))
fig.suptitle(f"AuxTel PWV seasonal variations — {version_run}",
             fontsize=13, y=1.005)

figfile1 = f"{pathfigs}/{prefix}_{version_run}_2panel_seasonal{figtype}"
fig.savefig(figfile1, bbox_inches="tight")
print(f"Saved: {figfile1}")
plt.show()

---
## 9. Compact single-panel with seasonal inset

For a one-column journal figure: main plot = full timeline + sine fit;
inset (upper left) = year-folded seasonal phase.

In [ ]:
fig2, ax_main = plt.subplots(figsize=(14, 6))

for filt in FILTERS_OF_INTEREST:
    sub = df_keep[df_keep["FILTER"] == filt]
    if len(sub) == 0:
        continue
    ax_main.errorbar(
        sub["MJD"], sub["PWV"], yerr=sub["PWV_err"],
        fmt=filter_marker[filt], color=filter_color[filt],
        markersize=3, linewidth=0, elinewidth=0.6, capsize=1.5,
        alpha=0.5, label=filter_label[filt], zorder=3
    )

ax_main.plot(mjd_curve, pwv_curve, color="black", lw=2, zorder=5,
             label=f"Annual sine: $A={A_fit:.2f}$, $|B|={abs(B_fit):.2f}$ mm")
ax_main.fill_between(mjd_curve, pwv_curve - pwv_band, pwv_curve + pwv_band,
                     alpha=0.15, color="black")

ax_main.set_xlabel("MJD", fontsize=12)
ax_main.set_ylabel("PWV [mm]", fontsize=12)
ax_main.set_title(
    "Precipitable Water Vapour — AuxTel/Spectractor, Cerro Pachon",
    fontsize=12, pad=26)
ax_main.set_xlim(mjd_min - 30, mjd_max + 30)
ax_main.set_ylim(PWVMIN, PWVMAX)
ax_main.legend(loc="upper right", fontsize=9, framealpha=0.85)
make_date_axis(ax_main, mjd_min, mjd_max, n_ticks=10)

# ── inset: seasonal phase ─────────────────────────────────────────────────────
ax_ins = ax_main.inset_axes([0.01, 0.55, 0.33, 0.41])

for yr in years_present:
    sub_yr = df_keep[df_keep["year"] == yr]
    for filt in FILTERS_OF_INTEREST:
        sub = sub_yr[sub_yr["FILTER"] == filt]
        if len(sub) == 0:
            continue
        ax_ins.errorbar(
            sub["doy_norm"], sub["PWV"], yerr=sub["PWV_err"],
            fmt=filter_marker[filt], color=year_color[yr],
            markersize=1, linewidth=0, elinewidth=0.35, capsize=0,
            alpha=0.45, zorder=3
        )

ax_ins.plot(t_phase_norm, pwv_phase, color="black", lw=1.5, zorder=5)
ax_ins.fill_between(t_phase_norm, pwv_phase - pwv_phase_band,
                    pwv_phase + pwv_phase_band, alpha=0.15, color="black")
ax_ins.errorbar(
    month_x, df_monthly["pwv_mean"], yerr=df_monthly["pwv_sem"],
    fmt="D", color="darkorange", markersize=5, elinewidth=1.1,
    capsize=3, zorder=6
)

ax_ins.set_xticks(month_ticks[::2])
ax_ins.set_xticklabels(MONTH_ABBR[::2], fontsize=7)
ax_ins.set_xlim(0, 1)
ax_ins.set_ylim(PWVMIN, PWVMAX)
ax_ins.tick_params(axis='y', labelsize=7)
ax_ins.set_ylabel("PWV [mm]", fontsize=8)
ax_ins.set_title("Seasonal (all years)", fontsize=8)
ax_ins.grid(True, alpha=0.3)

ins_hdl = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor=year_color[yr],
           markersize=5, label=str(yr))
    for yr in years_present
]
ax_ins.legend(handles=ins_hdl, fontsize=6, loc="upper right",
              ncol=2, framealpha=0.7, handlelength=0.8)

figfile2 = f"{pathfigs}/{prefix}_{version_run}_1panel_with_inset{figtype}"
fig2.savefig(figfile2, bbox_inches="tight")
print(f"Saved: {figfile2}")
plt.show()

---
## 10. Lomb-Scargle periodogram

Independent, non-parametric evidence for the annual period
(no sinusoidal assumption required).

In [ ]:
from astropy.timeseries import LombScargle

t_ls = df_keep["MJD"].values
y_ls = df_keep["PWV"].values
e_ls = df_keep["PWV_err"].values

mask_ls = np.isfinite(t_ls) & np.isfinite(y_ls) & np.isfinite(e_ls) & (e_ls > 0)
t_ls, y_ls, e_ls = t_ls[mask_ls], y_ls[mask_ls], e_ls[mask_ls]

ls = LombScargle(t_ls, y_ls, e_ls)

freq_grid   = np.linspace(1.0/1100, 1.0/2, 50_000)
power       = ls.power(freq_grid)
period_grid = 1.0 / freq_grid

fap_levels = [0.01, 0.001]
fap_powers = [ls.false_alarm_level(f) for f in fap_levels]

best_freq   = freq_grid[np.argmax(power)]
best_period = 1.0 / best_freq
print(f"Lomb-Scargle best period = {best_period:.1f} d  ({best_period/365.25:.3f} yr)")

In [ ]:
fig3, ax_ls = plt.subplots(figsize=(12, 5))

ax_ls.plot(period_grid, power, color="steelblue", lw=0.9, alpha=0.85)

for fap, pw, col in zip(fap_levels, fap_powers, ["orange", "red"]):
    ax_ls.axhline(pw, ls="--", color=col, lw=1.2,
                  label=f"FAP = {fap*100:.1f}%")

ax_ls.axvline(365.25, ls=":", color="black", lw=1.5, label="1 yr = 365.25 d")
ax_ls.axvline(best_period, ls="-", color="darkred", lw=1.8,
              label=f"Best period = {best_period:.0f} d")

ax_ls.set_xscale("log")
ax_ls.set_xlabel("Period [days]", fontsize=12)
ax_ls.set_ylabel("Lomb-Scargle power", fontsize=12)
ax_ls.set_title("Lomb-Scargle Periodogram — evidence for annual PWV cycle",
                fontsize=12)
ax_ls.legend(fontsize=10)

ax_ls2 = ax_ls.twiny()
ax_ls2.set_xlim(np.array(ax_ls.get_xlim()) / 365.25)
ax_ls2.set_xscale("log")
ax_ls2.set_xlabel("Period [yr]", fontsize=11)

figfile3 = f"{pathfigs}/{prefix}_{version_run}_lomb_scargle{figtype}"
fig3.savefig(figfile3, bbox_inches="tight")
print(f"Saved: {figfile3}")
plt.show()

---
## 11. Summary & export

In [ ]:
print("\n=== Sinusoidal fit summary ===")
print(f"  Model   : PWV(t) = A + B*sin(2pi t + phi)   [t in fractional years]")
print(f"  Dataset : {len(x_fit)} points after tight quality cuts")
print(f"  Period  : fixed = 1 yr")
print()
print(f"  A (offset)    = {A_fit:+.3f} +/- {A_err:.3f} mm")
print(f"  B (amplitude) = {B_fit:+.3f} +/- {B_err:.3f} mm")
print(f"  phi (phase)   = {phi_fit:+.3f} +/- {phi_err:.3f} rad")
print(f"  Min ~ month {month_min:.1f}  ({MONTH_ABBR[int(month_min)-1]})")
print(f"  Peak-to-peak  = {2*abs(B_fit):.2f} mm  ({2*abs(B_fit)/A_fit*100:.0f}% of mean)")
print()
print(f"=== Lomb-Scargle ===")
print(f"  Best period = {best_period:.0f} d = {best_period/365.25:.3f} yr")

In [ ]:
fit_results = {
    "version_run"         : version_run,
    "n_points"            : int(len(x_fit)),
    "A_mm"                : float(A_fit),
    "A_err_mm"            : float(A_err),
    "B_mm"                : float(B_fit),
    "B_err_mm"            : float(B_err),
    "phi_rad"             : float(phi_fit),
    "phi_err_rad"         : float(phi_err),
    "month_min"           : float(month_min),
    "peak_to_peak_mm"     : float(2*abs(B_fit)),
    "LS_best_period_days" : float(best_period),
}

outjson = f"{pathfigs}/pwv_annual_sine_fit_{version_run}.json"
with open(outjson, "w") as fh:
    json.dump(fit_results, fh, indent=2)
print(f"Saved: {outjson}")
print(json.dumps(fit_results, indent=2))

---
## 12. Seasonal PWV statistics

Seasons defined for the **Southern Hemisphere** (Cerro Pachon):

| Season | Months |
|--------|--------|
| Summer (DJF) | December, January, February |
| Autumn (MAM) | March, April, May |
| Winter (JJA) | June, July, August |
| Spring (SON) | September, October, November |

Statistics computed over **all years** and **all PWV-sensitive filters**
after tight quality cuts:
- **N** : number of individual measurements
- **Mean** : arithmetic mean [mm]
- **Median** : median [mm]
- **Std** : standard deviation [mm]
- **Wt. mean** : weighted mean (weights = 1/sigma^2) [mm]
- **Wt. std** : weighted standard deviation [mm]

In [ ]:
# ── Season definition — Southern Hemisphere ───────────────────────────────────
SEASON_MAP = {
    12: "Summer (DJF)",  1: "Summer (DJF)",  2: "Summer (DJF)",
     3: "Autumn (MAM)",  4: "Autumn (MAM)",  5: "Autumn (MAM)",
     6: "Winter (JJA)",  7: "Winter (JJA)",  8: "Winter (JJA)",
     9: "Spring (SON)", 10: "Spring (SON)", 11: "Spring (SON)",
}
SEASON_ORDER = ["Summer (DJF)", "Autumn (MAM)", "Winter (JJA)", "Spring (SON)"]

df_keep["season"] = df_keep["month"].map(SEASON_MAP)


def weighted_stats(vals, errs):
    """Weighted mean and weighted standard deviation."""
    w     = 1.0 / (errs**2 + 1e-12)
    mu_w  = np.average(vals, weights=w)
    std_w = np.sqrt(np.average((vals - mu_w)**2, weights=w))
    return mu_w, std_w


rows = []
for season in SEASON_ORDER:
    sub  = df_keep[df_keep["season"] == season]
    vals = sub["PWV"].values
    errs = sub["PWV_err"].values
    mask = np.isfinite(vals) & np.isfinite(errs) & (errs > 0)
    vals, errs = vals[mask], errs[mask]
    if len(vals) == 0:
        rows.append({"Season": season, "N": 0,
                     "Mean [mm]": np.nan, "Median [mm]": np.nan,
                     "Std [mm]": np.nan,
                     "Wt. mean [mm]": np.nan, "Wt. std [mm]": np.nan})
        continue
    mu_w, std_w = weighted_stats(vals, errs)
    rows.append({
        "Season"       : season,
        "N"            : len(vals),
        "Mean [mm]"    : np.mean(vals),
        "Median [mm]"  : np.median(vals),
        "Std [mm]"     : np.std(vals, ddof=1),
        "Wt. mean [mm]": mu_w,
        "Wt. std [mm]" : std_w,
    })

df_seasons = pd.DataFrame(rows).set_index("Season")

# ── styled display in notebook ────────────────────────────────────────────────
fmt_dict = {c: "{:.3f}" for c in df_seasons.columns if c != "N"}
fmt_dict["N"] = "{:d}"

display(
    df_seasons.style
    .format(fmt_dict)
    .set_caption(
        f"Seasonal PWV statistics — {version_run} (tight cuts, Southern Hemisphere)"
    )
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-weight", "bold"), ("font-size", "13px"),
                   ("text-align", "left"), ("padding-bottom", "8px")]},
        {"selector": "th",
         "props": [("background-color", "#dce6f1"), ("font-weight", "bold"),
                   ("text-align", "center")]},
        {"selector": "td",
         "props": [("text-align", "right"), ("padding", "4px 10px")]},
        {"selector": "tr:nth-child(even)",
         "props": [("background-color", "#f5f8fc")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#fff3cd")]},
    ])
    .bar(subset=["Mean [mm]", "Median [mm]", "Wt. mean [mm]"],
         color="#aec6e8", vmin=0)
)

In [ ]:
# ── LaTeX table ───────────────────────────────────────────────────────────────
header_latex = {
    "N"            : r"$N$",
    "Mean [mm]"    : r"$\langle\mathrm{PWV}\rangle$ [mm]",
    "Median [mm]"  : r"$\widetilde{\mathrm{PWV}}$ [mm]",
    "Std [mm]"     : r"$\sigma$ [mm]",
    "Wt. mean [mm]": r"$\bar{w}\mathrm{PWV}$ [mm]",
    "Wt. std [mm]" : r"$\sigma_w$ [mm]",
}

col_spec = "l" + "r" * len(df_seasons.columns)

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(
    r"\caption{Seasonal statistics of the precipitable water vapour (PWV) "
    r"at Cerro Pach\'{o}n measured by AuxTel/Spectractor after tight quality cuts. "
    r"Seasons are defined for the Southern Hemisphere (Summer = DJF, "
    r"Autumn = MAM, Winter = JJA, Spring = SON). "
    r"$\langle\mathrm{PWV}\rangle$: arithmetic mean; "
    r"$\widetilde{\mathrm{PWV}}$: median; "
    r"$\sigma$: standard deviation; "
    r"$\bar{w}\mathrm{PWV}$ and $\sigma_w$: measurement-uncertainty-weighted "
    r"mean and standard deviation.}"
)
lines.append(r"\label{tab:pwv_seasonal}")
lines.append(rf"\begin{{tabular}}{{{col_spec}}}")
lines.append(r"\hline\hline")

header_cells = ["Season"] + [header_latex.get(c, c) for c in df_seasons.columns]
lines.append(" & ".join(header_cells) + r" \\")
lines.append(r"\hline")

for season, row in df_seasons.iterrows():
    cells = [season.replace("(", "(").replace(")", ")")]  # keep as-is, no special chars
    for col in df_seasons.columns:
        if col == "N":
            cells.append(str(int(row[col])))
        else:
            cells.append(f"{row[col]:.3f}")
    lines.append(" & ".join(cells) + r" \\")

lines.append(r"\hline")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex_str = "\n".join(lines)

# print for copy-paste
print(latex_str)

# save to file
latex_file = f"{pathfigs}/{prefix}_{version_run}_seasonal_stats_table.tex"
with open(latex_file, "w") as fh:
    fh.write(latex_str + "\n")
print(f"\nSaved LaTeX table: {latex_file}")